#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import densenet121, DenseNet121_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [2]:
DATA_DIR = r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification"

WEIGHTS_DIR = "../weights"

BATCH_SIZE = 32
EPOCHS_FREEZE = 8
EPOCHS_FINETUNE = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
weights = DenseNet121_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])


In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
print("Train class_to_idx:", train_dataset.class_to_idx)
print("Val class_to_idx:", val_dataset.class_to_idx)

Train class_to_idx: {'Healthy': 0, 'Spider Mites': 1, 'leaf blight': 2, 'leaf spot': 3}
Val class_to_idx: {'Healthy': 0, 'Spider Mites': 1, 'leaf blight': 2, 'leaf spot': 3}


In [6]:
model = densenet121(weights=weights)

in_features = model.classifier.in_features

model.classifier = nn.Linear(in_features, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [7]:
criterion = nn.CrossEntropyLoss()

In [8]:
def train_one_epoch(loader, epoch, epochs, optimizer):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [9]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating "):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

#### Training 1

In [10]:
for param in model.features.parameters():
    param.requires_grad = False


In [11]:
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)


In [12]:

print("\n-----------Starting Phase 1 Training-----------\n")

for epoch in range(EPOCHS_FREEZE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_FREEZE, optimizer)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")




-----------Starting Phase 1 Training-----------



Validating : 100%|██████████| 8/8 [01:18<00:00,  9.76s/it]



Epoch [1/8]
Train Loss: 31.7716 | Train Acc: 0.6292
Val Loss: 6.6135 | Val Acc: 0.7292
Precision: 0.7526 | Recall: 0.7292 | F1: 0.7311




Validating : 100%|██████████| 8/8 [01:16<00:00,  9.51s/it]



Epoch [2/8]
Train Loss: 18.3151 | Train Acc: 0.8542
Val Loss: 4.9244 | Val Acc: 0.7833
Precision: 0.7982 | Recall: 0.7833 | F1: 0.7839




...


Epoch [6/8]
Train Loss: 10.4198 | Train Acc: 0.8927
Val Loss: 3.2128 | Val Acc: 0.8583
Precision: 0.8597 | Recall: 0.8583 | F1: 0.8587




Validating : 100%|██████████| 8/8 [01:12<00:00,  9.10s/it]



Epoch [7/8]
Train Loss: 9.8345 | Train Acc: 0.8938
Val Loss: 3.0696 | Val Acc: 0.8750
Precision: 0.8744 | Recall: 0.8750 | F1: 0.8746




Validating : 100%|██████████| 8/8 [01:05<00:00,  8.16s/it]


Epoch [8/8]
Train Loss: 9.0212 | Train Acc: 0.9146
Val Loss: 3.0362 | Val Acc: 0.8958
Precision: 0.9016 | Recall: 0.8958 | F1: 0.8963




#### Training 2

In [13]:
for name, param in model.features.named_parameters():
    if "denseblock4" in name:
        param.requires_grad = True


In [14]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)


In [15]:

print("\n-----------Starting Phase 2 Training-----------\n")

for epoch in range(EPOCHS_FREEZE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_FREEZE, optimizer)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")




-----------Starting Phase 2 Training-----------



Validating : 100%|██████████| 8/8 [00:43<00:00,  5.45s/it]



Epoch [1/8]
Train Loss: 7.3712 | Train Acc: 0.9281
Val Loss: 2.3938 | Val Acc: 0.8917
Precision: 0.8952 | Recall: 0.8917 | F1: 0.8921




Validating : 100%|██████████| 8/8 [01:12<00:00,  9.09s/it]



Epoch [2/8]
Train Loss: 4.8416 | Train Acc: 0.9510
Val Loss: 2.1465 | Val Acc: 0.9125
Precision: 0.9231 | Recall: 0.9125 | F1: 0.9132




...


Epoch [6/8]
Train Loss: 2.2095 | Train Acc: 0.9750
Val Loss: 1.5243 | Val Acc: 0.9250
Precision: 0.9267 | Recall: 0.9250 | F1: 0.9254




Validating : 100%|██████████| 8/8 [00:42<00:00,  5.32s/it]



Epoch [7/8]
Train Loss: 1.5000 | Train Acc: 0.9896
Val Loss: 1.5388 | Val Acc: 0.9292
Precision: 0.9332 | Recall: 0.9292 | F1: 0.9299




Validating : 100%|██████████| 8/8 [00:41<00:00,  5.22s/it]


Epoch [8/8]
Train Loss: 1.8282 | Train Acc: 0.9812
Val Loss: 1.5749 | Val Acc: 0.9250
Precision: 0.9305 | Recall: 0.9250 | F1: 0.9254




#### Training 3

In [16]:
for name, param in model.features.named_parameters():
    if "denseblock3" in name:
        param.requires_grad = True


In [17]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)


In [18]:

print("\n-----------Starting Phase 3 Training-----------\n")

for epoch in range(EPOCHS_FREEZE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_FREEZE, optimizer)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")




-----------Starting Phase 3 Training-----------



Validating : 100%|██████████| 8/8 [01:14<00:00,  9.30s/it]



Epoch [1/8]
Train Loss: 1.0126 | Train Acc: 0.9927
Val Loss: 1.4661 | Val Acc: 0.9417
Precision: 0.9454 | Recall: 0.9417 | F1: 0.9418




Validating : 100%|██████████| 8/8 [01:15<00:00,  9.41s/it]



Epoch [2/8]
Train Loss: 0.8126 | Train Acc: 0.9917
Val Loss: 1.2383 | Val Acc: 0.9542
Precision: 0.9576 | Recall: 0.9542 | F1: 0.9545




...


Epoch [6/8]
Train Loss: 0.3290 | Train Acc: 0.9990
Val Loss: 1.2661 | Val Acc: 0.9542
Precision: 0.9554 | Recall: 0.9542 | F1: 0.9543




Validating : 100%|██████████| 8/8 [01:13<00:00,  9.17s/it]



Epoch [7/8]
Train Loss: 0.3187 | Train Acc: 0.9979
Val Loss: 1.3172 | Val Acc: 0.9458
Precision: 0.9476 | Recall: 0.9458 | F1: 0.9462




Validating : 100%|██████████| 8/8 [01:16<00:00,  9.56s/it]


Epoch [8/8]
Train Loss: 0.2024 | Train Acc: 1.0000
Val Loss: 1.3075 | Val Acc: 0.9500
Precision: 0.9510 | Recall: 0.9500 | F1: 0.9502




#### Training 4 (Fine-Tuning)

In [19]:
for param in model.parameters():
    param.requires_grad = True


In [20]:
optimizer = optim.Adam(model.parameters(), lr=1e-5)


In [21]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(EPOCHS_FINETUNE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_FINETUNE, optimizer)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FINETUNE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "DenseNet121.pth"))


-----------Starting Fine-tuning Stage-----------



Validating : 100%|██████████| 8/8 [00:41<00:00,  5.14s/it]



Epoch [1/20]
Train Loss: 0.2471 | Train Acc: 0.9990
Val Loss: 1.2354 | Val Acc: 0.9583
Precision: 0.9589 | Recall: 0.9583 | F1: 0.9584




Validating : 100%|██████████| 8/8 [01:15<00:00,  9.44s/it]



Epoch [2/20]
Train Loss: 0.1891 | Train Acc: 1.0000
Val Loss: 1.2729 | Val Acc: 0.9542
Precision: 0.9555 | Recall: 0.9542 | F1: 0.9544




...


Epoch [18/20]
Train Loss: 0.0788 | Train Acc: 1.0000
Val Loss: 1.1735 | Val Acc: 0.9750
Precision: 0.9767 | Recall: 0.9750 | F1: 0.9751




Validating : 100%|██████████| 8/8 [01:15<00:00,  9.43s/it]



Epoch [19/20]
Train Loss: 0.1378 | Train Acc: 0.9990
Val Loss: 1.1233 | Val Acc: 0.9708
Precision: 0.9730 | Recall: 0.9708 | F1: 0.9711




Validating : 100%|██████████| 8/8 [01:08<00:00,  8.61s/it]


Epoch [20/20]
Train Loss: 0.1632 | Train Acc: 0.9990
Val Loss: 1.2193 | Val Acc: 0.9750
Precision: 0.9767 | Recall: 0.9750 | F1: 0.9751




#### Prediction Sample

In [23]:
checkpoint = torch.load(r'../weights/DenseNet121.pth')

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


# Example usage:
prediction = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_13.jpg")
prediction1 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Blight\leaf blight_val_15.jpg")
prediction2 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Spot\leaf spot_val_14.jpg")
prediction3 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_24.jpg")
print(f"Healthy Predicted class: {prediction}")
print(f"Leaf Blight Predicted class: {prediction1}")
print(f"Leaf Spot Predicted class: {prediction2}")
print(f"Spider Mites Predicted class: {prediction3}")

Healthy Predicted class: Healthy
Leaf Blight Predicted class: leaf blight
Leaf Spot Predicted class: leaf spot
Spider Mites Predicted class: Spider Mites
